# 06 — Năm phương pháp và ba thí nghiệm

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Lựa chọn bằng CV

Baseline, Ridge, XGBoost, ARIMA và GRU là năm phương pháp chính. GRU chỉ thử L=20/60. SARIMAX là đối sánh mở rộng; XGBoost giữ cùng tham số khi thêm dữ liệu. Ô sau chạy lại huấn luyện và ghi artefact.

In [ ]:
from train import run_training
summary=run_training()
print(json.dumps(summary,ensure_ascii=False,indent=2))

## Thí nghiệm 1: bảng chính

Đọc CV trước để biết lựa chọn. MAE/RMSE giá thấp là tốt; đúng hướng/dung sai cao là tốt. Baseline đứng giá nên đúng hướng bằng 0 trên phiên thực tế có đổi giá.

In [ ]:
cols=['model','cv_rmse_mean','cv_rmse_std','mae','rmse','directional_accuracy_nonflat_percent','tolerance_accuracy_1_percent']
display(pd.read_csv(ROOT/'reports/main_benchmark.csv')[cols])
display(Image(filename=str(ROOT/'figures/main_benchmark_prices.png')))

## Thí nghiệm 2: thêm dữ liệu

32 → 38 → 42 biến. Đọc riêng sai số log return, giá và dấu; không mặc định dữ liệu nhiều hơn sẽ tốt hơn.

In [ ]:
display(pd.read_csv(ROOT/'reports/ablation.csv'))
display(pd.read_csv(ROOT/'reports/statistical_extension.csv'))
display(Image(filename=str(ROOT/'figures/ablation_metrics.png')))

## Thí nghiệm 3: bộ nhớ GRU

L=20 gần một tháng giao dịch; L=60 gần ba tháng. Chọn cửa sổ có CV RMSE thấp hơn, không chọn theo cột Test.

In [ ]:
display(pd.read_csv(ROOT/'reports/lookback.csv'))
display(Image(filename=str(ROOT/'figures/lookback_metrics.png')))